In [64]:
import os
import sys
import pandas as pd
from ydata_profiling import ProfileReport
current = os.getcwd()
path_to_root = os.path.join (current, '../')
abs_path = os.path.abspath(path_to_root)
sys.path.append(abs_path)
import config
import ijson
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

client = config.create_minio_client()

[Bucket('aws'), Bucket('azure'), Bucket('azure-clean'), Bucket('google'), Bucket('google-clean')]


In [65]:
# object_name = "AmazonEC2.json"
# object_name = "AmazonS3.json"
# object_name = "AmazonRDS.json"
object_name = "AmazonEKS.json"
# object_name = "AmazonVPC.json"

TARGET_RECORDS = 100000 
sample_products = []

response = client.get_object(config.PROVIDERS.get("aws").get("bucket"), object_name=object_name)

try:
    # Το ijson διαβάζει κατευθείαν από το stream byte-byte
    parser = ijson.kvitems(response, 'products')
    
    count = 0
    for sku, product_data in parser:
        #Mε την προοπτική να δημιουργεί πεδίο με sku με την αντίστοιχη τιμή μέσα στο dict αλλά αυτό ήδη υπάρχει
        # product_data['sku'] = sku
        sample_products.append(product_data)
        
        count += 1
        if count >= TARGET_RECORDS:
            break
            
    print(f"Downloaded  {len(sample_products)} records μέσω streaming.")

finally:
    response.close()
    response.release_conn()

# Μετατροπή σε αρχικό DataFrame
df_products = pd.json_normalize(sample_products)
print(f"DataFrame: Rows = {df_products.shape[0]}, Columns = {df_products.shape[1]}")
df_products.head()


Downloaded  20394 records μέσω streaming.
DataFrame: Rows = 20394, Columns = 18


,sku,productFamily,attributes.servicecode,attributes.location,attributes.locationType,attributes.usagetype,attributes.operation,attributes.eksproducttype,attributes.instancetype,attributes.regionCode,attributes.servicename,attributes.tiertype,attributes.capabilitytype,attributes.ekscapabilityunits,attributes.tenancy,attributes.memorytype,attributes.cputype,attributes.storagetype
0,A53K5SWT3G3CRHF5,Compute,AmazonEKS,Asia Pacific (Malaysia),AWS Region,APS7-EKS-Auto:r8i.8xlarge-management-hours,EKSAutoUsage,AutoMode,r8i.8xlarge,ap-southeast-5,Amazon Elastic Container Service for Kubernetes,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,TCP2YWGAHGPW67N9,Compute,AmazonEKS,South America (Sao Paulo),AWS Region,SAE1-EKS-Auto:m6g.metal-management-hours,EKSAutoUsage,AutoMode,m6g.metal,sa-east-1,Amazon Elastic Container Service for Kubernetes,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,MCK76ZZJZMWS7HDE,Compute,AmazonEKS,Middle East (Bahrain),AWS Region,MES1-EKS-Auto:r5d.xlarge-management-hours,EKSAutoUsage,AutoMode,r5d.xlarge,me-south-1,Amazon Elastic Container Service for Kubernetes,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,FQKX5BN95S5F5T2E,Compute,AmazonEKS,Asia Pacific (Sydney),AWS Region,APS2-EKS-Auto:m5.4xlarge-management-hours,EKSAutoUsage,AutoMode,m5.4xlarge,ap-southeast-2,Amazon Elastic Container Service for Kubernetes,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,USZRSB47YGRTASDR,Compute,AmazonEKS,Asia Pacific (Malaysia),AWS Region,APS7-EKS-Auto:i8ge.large-management-hours,EKSAutoUsage,AutoMode,i8ge.large,ap-southeast-5,Amazon Elastic Container Service for Kubernetes,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [66]:
# Στο πάνω κελί είχα μία λίστα η οποία είχε μέσα n sku, μαζί με όλα τα attributes τουσ.
# Τώρα στο βήμα αυτό κάνω access την λίστα και απομονώνω σε ένα set μόνο τον κωδικό των n sku (η επιλογή set βασίζεται στην γρήγορη αναζήτηση)
target_skus = {p['sku'] for p in sample_products}

# Αυτή θα είναι η αντίστοιχη sample products του πάνω βήματος. Θα κρατάει τα ζευγάρια sku, με στοιχεία πληρωμής, και μετά θα την κάνουμε dataframe
terms_list = []

parser = client.get_object(config.PROVIDERS.get("aws").get("bucket"), object_name=object_name)

try:
    # Πάμε βαθύτερα και από το λεξικό terms θα στοχεύσουμε μόνο στις On-Demand υπηρεσίες.
    terms_parser = ijson.kvitems(parser, 'terms.OnDemand')

    # Προφανώς δεν τα θέλω όλα!!Μόνο εκείνα των οποίων το sku βρίσκεται στο set το οποίο δημιούργησα
    for sku, term_offers in terms_parser:
        if sku not in target_skus:
            continue
        
        # Αποθηκεύουμε το sku και ολόκληρο το raw λεξικό των terms του
        terms_list.append({
            'skuNew': sku,
            'termsOndemand': term_offers
        })
        
        # Aπλά για να βεβαιωθώ ότι όσα products πήρα άλλες τόσες και οι τιμές 
        if len(terms_list) >= len(target_skus):
            break
            
    print(f"Downloaded {len(terms_list)} matching terms μέσω streaming.")

finally:
    parser.close()
    parser.release_conn()


df_terms = pd.DataFrame(terms_list)
print(f"Terms DataFrame: Rows = {df_terms.shape[0]}, Columns = {df_terms.shape[1]}")
df_terms.head()

Downloaded 20394 matching terms μέσω streaming.
Terms DataFrame: Rows = 20394, Columns = 2


,skuNew,termsOndemand
0,A53K5SWT3G3CRHF5,"{'A53K5SWT3G3CRHF5.JRTCKXETXF': {'offerTermCode': 'JRTCKXETXF', 'sku': 'A53K5SWT3G3CRHF5', 'effectiveDate': '2026-05-01T00:00:00Z', 'priceDimensions': {'A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7': {'rateCode': 'A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7', 'description': '$0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia)', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'hours', 'pricePerUnit': {'USD': '0.2734800000'}, 'appliesTo': []}}, 'termAttributes': {}}}"
1,TCP2YWGAHGPW67N9,"{'TCP2YWGAHGPW67N9.JRTCKXETXF': {'offerTermCode': 'JRTCKXETXF', 'sku': 'TCP2YWGAHGPW67N9', 'effectiveDate': '2026-05-01T00:00:00Z', 'priceDimensions': {'TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7': {'rateCode': 'TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7', 'description': '$0.47002 per hour for EKS Auto Mode management of m6g.metal in South America (Sao Paulo)', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'hours', 'pricePerUnit': {'USD': '0.4700200000'}, 'appliesTo': []}}, 'termAttributes': {}}}"
2,MCK76ZZJZMWS7HDE,"{'MCK76ZZJZMWS7HDE.JRTCKXETXF': {'offerTermCode': 'JRTCKXETXF', 'sku': 'MCK76ZZJZMWS7HDE', 'effectiveDate': '2026-05-01T00:00:00Z', 'priceDimensions': {'MCK76ZZJZMWS7HDE.JRTCKXETXF.6YS6EN2CT7': {'rateCode': 'MCK76ZZJZMWS7HDE.JRTCKXETXF.6YS6EN2CT7', 'description': '$0.04224 per hour for EKS Auto Mode management of r5d.xlarge in Middle East (Bahrain)', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'hours', 'pricePerUnit': {'USD': '0.0422400000'}, 'appliesTo': []}}, 'termAttributes': {}}}"
3,FQKX5BN95S5F5T2E,"{'FQKX5BN95S5F5T2E.JRTCKXETXF': {'offerTermCode': 'JRTCKXETXF', 'sku': 'FQKX5BN95S5F5T2E', 'effectiveDate': '2026-05-01T00:00:00Z', 'priceDimensions': {'FQKX5BN95S5F5T2E.JRTCKXETXF.6YS6EN2CT7': {'rateCode': 'FQKX5BN95S5F5T2E.JRTCKXETXF.6YS6EN2CT7', 'description': '$0.11520 per hour for EKS Auto Mode management of m5.4xlarge in Asia Pacific (Sydney)', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'hours', 'pricePerUnit': {'USD': '0.1152000000'}, 'appliesTo': []}}, 'termAttributes': {}}}"
4,USZRSB47YGRTASDR,"{'USZRSB47YGRTASDR.JRTCKXETXF': {'offerTermCode': 'JRTCKXETXF', 'sku': 'USZRSB47YGRTASDR', 'effectiveDate': '2026-05-01T00:00:00Z', 'priceDimensions': {'USZRSB47YGRTASDR.JRTCKXETXF.6YS6EN2CT7': {'rateCode': 'USZRSB47YGRTASDR.JRTCKXETXF.6YS6EN2CT7', 'description': '$0.03074 per hour for EKS Auto Mode management of i8ge.large in Asia Pacific (Malaysia)', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'hours', 'pricePerUnit': {'USD': '0.0307400000'}, 'appliesTo': []}}, 'termAttributes': {}}}"


Θα δουλέψουμε αρχικά με το 2ο dataframe το οποίο περιέχει τα δεδομένα τιμολόγησης. Κρίνονται απαραίτητες 4 ενέργειες
- Άνοιγμα 2η στήλης και άπλωμα δεδομένων
- Αντιστοιχία εσωτερικού sku με αυτό που έβαλα εγώ, και πέταμα μίας στήλης εκ των 2
- Μελέτη για εντοπισμό καθολικών στηλών 
- Αφαίρεση περιττών στηλών
- Κατανόηση pricing και αντιστοίχιση με τους άλλους παρόχους

In [67]:
# Πάιρνω την πρώτη εγγραφή προκειμένου να κάνω έναν έλεγχο των πεδίων
sample_row = df_terms.iloc[0]
print("SKU:", sample_row['skuNew'])

raw_dict = sample_row['termsOndemand']

# Βρίσκουμε το κλειδί (το σύνθετο hash, π.χ. SKU.OfferTermCode)
offer_hash_key = list(raw_dict.keys())[0]
offer_content = raw_dict[offer_hash_key]

print("\n--- Περιεχόμενα προσφοράς (Offer Content) ---")
for k, v in offer_content.items():
    # if k != 'priceDimensions':
    print(f"{k}: {v}")

SKU: A53K5SWT3G3CRHF5

--- Περιεχόμενα προσφοράς (Offer Content) ---
offerTermCode: JRTCKXETXF
sku: A53K5SWT3G3CRHF5
effectiveDate: 2026-05-01T00:00:00Z
priceDimensions: {'A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7': {'rateCode': 'A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7', 'description': '$0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia)', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'hours', 'pricePerUnit': {'USD': '0.2734800000'}, 'appliesTo': []}}
termAttributes: {}


Ο παρακάτω κώδικας υλοποιεί ένα μέρος του πρώτου βήματος. Πετάει ένα περιττό hash το οποίο ήταν sku + offerCode, και ανοίγει εν μέρη το λεξικό termsOnDemand το οποίο κατασκεύσαμε όταν πήραμε τα δεδομένα και τα μετατρέψαμε σε datframe. Συγκεκριμένα εξάγει μερικά πεδία και τα κάνει κανονικές στήλες. Το απευθείας normalize δοκιμάστηκε και απετύχε, οπότε και προχωρήσαμε με ένα for loop.

In [68]:
flattened_list = []

for idx, row in df_terms.iterrows():
    skuDefaultValue = row['skuNew']
    raw_dict = row['termsOndemand']
    
    # ΜΠετάω το αρχικό κλειδί το οποίο είχε το dict. Περισσοτερα στην αναφορά
    for hash_key, offer_content in raw_dict.items():
        
        # Εξάγω τα πεδία ένα - ένα και φτιάχνω μία δική μου δομή πιο υύκολη στην ανάλυση
        item = {
            'skuNew': skuDefaultValue,
            'offerTermCode': offer_content.get('offerTermCode'),
            'sku': offer_content.get('sku'),
            'effectiveDate': offer_content.get('effectiveDate'),
            'termAttributes': offer_content.get('termAttributes'),
            'priceDimensions': offer_content.get('priceDimensions') # Το αφήνουμε λεξικό!
        }
        flattened_list.append(item)

df_terms = pd.DataFrame(flattened_list)

print(df_terms.columns)
df_terms.head(2)

Index(['skuNew', 'offerTermCode', 'sku', 'effectiveDate', 'termAttributes',
       'priceDimensions'],
      dtype='object')


,skuNew,offerTermCode,sku,effectiveDate,termAttributes,priceDimensions
0,A53K5SWT3G3CRHF5,JRTCKXETXF,A53K5SWT3G3CRHF5,2026-05-01T00:00:00Z,{},"{'A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7': {'rateCode': 'A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7', 'description': '$0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia)', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'hours', 'pricePerUnit': {'USD': '0.2734800000'}, 'appliesTo': []}}"
1,TCP2YWGAHGPW67N9,JRTCKXETXF,TCP2YWGAHGPW67N9,2026-05-01T00:00:00Z,{},"{'TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7': {'rateCode': 'TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7', 'description': '$0.47002 per hour for EKS Auto Mode management of m6g.metal in South America (Sao Paulo)', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'hours', 'pricePerUnit': {'USD': '0.4700200000'}, 'appliesTo': []}}"


Επόμενο βήμα είναι το περαιτέρω άνοιγμα του λεξικού το οποίο κρύβει μέσα τις πληροφοριες τιμολόγησης. Συγκεκριμένα το priceDimensions. Όπως φαίνεται και από το πάνω αποτέλεσμα του κελιού, μέσα στο λεξικό αυτό υπάρχει άλλο ένα περίεργο hash - κλειδί, το οποίο όμως όπως και στο παραπάνω κελί θα απορρίψουμε. Ο κώδικας λοιπόν ακολουθεί την ίδια τακτική: διατρέχει γραμμή γραμμή, προσπερνάει το περίεργο αυτό, και τραβάει μόνο τα πραγματικά δεδομένα.

In [69]:
df_terms.head()

,skuNew,offerTermCode,sku,effectiveDate,termAttributes,priceDimensions
0,A53K5SWT3G3CRHF5,JRTCKXETXF,A53K5SWT3G3CRHF5,2026-05-01T00:00:00Z,{},"{'A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7': {'rateCode': 'A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7', 'description': '$0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia)', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'hours', 'pricePerUnit': {'USD': '0.2734800000'}, 'appliesTo': []}}"
1,TCP2YWGAHGPW67N9,JRTCKXETXF,TCP2YWGAHGPW67N9,2026-05-01T00:00:00Z,{},"{'TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7': {'rateCode': 'TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7', 'description': '$0.47002 per hour for EKS Auto Mode management of m6g.metal in South America (Sao Paulo)', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'hours', 'pricePerUnit': {'USD': '0.4700200000'}, 'appliesTo': []}}"
2,MCK76ZZJZMWS7HDE,JRTCKXETXF,MCK76ZZJZMWS7HDE,2026-05-01T00:00:00Z,{},"{'MCK76ZZJZMWS7HDE.JRTCKXETXF.6YS6EN2CT7': {'rateCode': 'MCK76ZZJZMWS7HDE.JRTCKXETXF.6YS6EN2CT7', 'description': '$0.04224 per hour for EKS Auto Mode management of r5d.xlarge in Middle East (Bahrain)', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'hours', 'pricePerUnit': {'USD': '0.0422400000'}, 'appliesTo': []}}"
3,FQKX5BN95S5F5T2E,JRTCKXETXF,FQKX5BN95S5F5T2E,2026-05-01T00:00:00Z,{},"{'FQKX5BN95S5F5T2E.JRTCKXETXF.6YS6EN2CT7': {'rateCode': 'FQKX5BN95S5F5T2E.JRTCKXETXF.6YS6EN2CT7', 'description': '$0.11520 per hour for EKS Auto Mode management of m5.4xlarge in Asia Pacific (Sydney)', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'hours', 'pricePerUnit': {'USD': '0.1152000000'}, 'appliesTo': []}}"
4,USZRSB47YGRTASDR,JRTCKXETXF,USZRSB47YGRTASDR,2026-05-01T00:00:00Z,{},"{'USZRSB47YGRTASDR.JRTCKXETXF.6YS6EN2CT7': {'rateCode': 'USZRSB47YGRTASDR.JRTCKXETXF.6YS6EN2CT7', 'description': '$0.03074 per hour for EKS Auto Mode management of i8ge.large in Asia Pacific (Malaysia)', 'beginRange': '0', 'endRange': 'Inf', 'unit': 'hours', 'pricePerUnit': {'USD': '0.0307400000'}, 'appliesTo': []}}"


In [70]:
itemsList = []

for idx, row in df_terms.iterrows():

    # Κατασκευάζουμε εξ'ολοκλήρου νεό dataframe. Δεν κάνουμε ενέργειες πάνω στον υπάρχον. Οπότε φτιάχνουμε γραμμή γραμμή με τα πεδία που έχουμε. Για αυτό και τα εξάξουμε ένα - ένα
    skuNew = row['skuNew']
    offerTermCode = row['offerTermCode']
    skuDefaultValue = row['sku']
    effectiveDate = row['effectiveDate']
    termAttributes = row['termAttributes']
    
    # Παίρνουμε το λεξικό του priceDimensions
    priceDimensionDict = row['priceDimensions']

    # Κοιτάζει εάν όντως το priceDimensions είναι λεξικό. Αν δεν είναι δεν μπαίνει καν μέσα στο loop. Έτσι και δεν κρασάρει, και αποφεύγω να δημιουργήσω γραμμές στις οποίες τα δεδομένα είναι ελλιπή
    if isinstance(priceDimensionDict, dict):
        # ΔΌπως και στο πάνω κελί, με τον τρόπο αυτό αγνοούμε το εσωτετικό xxx.xxx.xxx
        for dimensionsDict_hash_key, dimensionsDict_content in priceDimensionDict.items():
            
            # Το pricePerUnit είναι και αυτό με την σειρά του λεξικού οποτε πριν εφαρμόσω την μέθοδο get προσέχω για να βεβαιωθώ ότι το βρήκα και δεν έπεσα στην περίπτωση "κακών" δεδομένων
            price_per_unit_dict = dimensionsDict_content.get('pricePerUnit', {})
            usd_price = price_per_unit_dict.get('USD') if isinstance(price_per_unit_dict, dict) else None
            
            item = {
                'skuNew': skuNew,
                'offerTermCode': offerTermCode,
                'sku': skuDefaultValue,
                'effectiveDate': effectiveDate,
                'termAttributes': termAttributes,
                'rateCode': dimensionsDict_content.get('rateCode'),
                'description': dimensionsDict_content.get('description'),
                'beginRange': dimensionsDict_content.get('beginRange'),
                'endRange': dimensionsDict_content.get('endRange'),
                'unit': dimensionsDict_content.get('unit'),
                'priceUSD': usd_price,   # Aυτό το πεδίο μέσω του ελέγχου που κάναμε πιο πάνω, ή θα είναι None ή θα έχει κάποια τιμή. Οπότε θα το χρησιμοποιήσω μετά για έλεγχω
                'appliesTo': dimensionsDict_content.get('appliesTo')
            }
            itemsList.append(item)

df_terms_final = pd.DataFrame(itemsList)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (20530, 12)


,skuNew,offerTermCode,sku,effectiveDate,termAttributes,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,A53K5SWT3G3CRHF5,JRTCKXETXF,A53K5SWT3G3CRHF5,2026-05-01T00:00:00Z,{},A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7,$0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia),0,Inf,hours,0.2734800000,[]
1,TCP2YWGAHGPW67N9,JRTCKXETXF,TCP2YWGAHGPW67N9,2026-05-01T00:00:00Z,{},TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7,$0.47002 per hour for EKS Auto Mode management of m6g.metal in South America (Sao Paulo),0,Inf,hours,0.4700200000,[]
2,MCK76ZZJZMWS7HDE,JRTCKXETXF,MCK76ZZJZMWS7HDE,2026-05-01T00:00:00Z,{},MCK76ZZJZMWS7HDE.JRTCKXETXF.6YS6EN2CT7,$0.04224 per hour for EKS Auto Mode management of r5d.xlarge in Middle East (Bahrain),0,Inf,hours,0.0422400000,[]
3,FQKX5BN95S5F5T2E,JRTCKXETXF,FQKX5BN95S5F5T2E,2026-05-01T00:00:00Z,{},FQKX5BN95S5F5T2E.JRTCKXETXF.6YS6EN2CT7,$0.11520 per hour for EKS Auto Mode management of m5.4xlarge in Asia Pacific (Sydney),0,Inf,hours,0.1152000000,[]
4,USZRSB47YGRTASDR,JRTCKXETXF,USZRSB47YGRTASDR,2026-05-01T00:00:00Z,{},USZRSB47YGRTASDR.JRTCKXETXF.6YS6EN2CT7,$0.03074 per hour for EKS Auto Mode management of i8ge.large in Asia Pacific (Malaysia),0,Inf,hours,0.0307400000,[]


##### Επεξεργασία πίνακα με τα δεδομένα τιμολόγησης 
1. Λίστα appliesTo ή οποία μπορεί να φαίνεται κενή, αλλά θα την φροντίσουμε με explode και reindex για κάθε ενδεχόμενο. Επίσης το το πεδίο termAtrributes το οποίο είναι ένα dict κενό, θα το αφαιρέσουμε καθώς μετά από μελέτη του documentaion κρίθηκε άχρηστο αφ'ης στιγμής κρατάμε μόνο On-Demand εγγραφές Δεν χρειάζονται περίεργα loop ή εντολές σύνθετες. Απλή χρήση των 2 εντολών. 

In [ ]:
df_terms_final = df_terms_final.explode('appliesTo').reset_index(drop=True)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (20530, 12)


,skuNew,offerTermCode,sku,effectiveDate,termAttributes,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,A53K5SWT3G3CRHF5,JRTCKXETXF,A53K5SWT3G3CRHF5,2026-05-01T00:00:00Z,{},A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7,$0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia),0,Inf,hours,0.2734800000,NaN
1,TCP2YWGAHGPW67N9,JRTCKXETXF,TCP2YWGAHGPW67N9,2026-05-01T00:00:00Z,{},TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7,$0.47002 per hour for EKS Auto Mode management of m6g.metal in South America (Sao Paulo),0,Inf,hours,0.4700200000,NaN
2,MCK76ZZJZMWS7HDE,JRTCKXETXF,MCK76ZZJZMWS7HDE,2026-05-01T00:00:00Z,{},MCK76ZZJZMWS7HDE.JRTCKXETXF.6YS6EN2CT7,$0.04224 per hour for EKS Auto Mode management of r5d.xlarge in Middle East (Bahrain),0,Inf,hours,0.0422400000,NaN
3,FQKX5BN95S5F5T2E,JRTCKXETXF,FQKX5BN95S5F5T2E,2026-05-01T00:00:00Z,{},FQKX5BN95S5F5T2E.JRTCKXETXF.6YS6EN2CT7,$0.11520 per hour for EKS Auto Mode management of m5.4xlarge in Asia Pacific (Sydney),0,Inf,hours,0.1152000000,NaN
4,USZRSB47YGRTASDR,JRTCKXETXF,USZRSB47YGRTASDR,2026-05-01T00:00:00Z,{},USZRSB47YGRTASDR.JRTCKXETXF.6YS6EN2CT7,$0.03074 per hour for EKS Auto Mode management of i8ge.large in Asia Pacific (Malaysia),0,Inf,hours,0.0307400000,NaN


In [72]:
if 'termAttributes' in df_terms_final.columns:
    df_terms_final = df_terms_final.drop(columns=['termAttributes'])

print (f"Dimension (rows,cols): {df_terms_final.shape}")
df_terms_final.head()


Dimension (rows,cols): (20530, 11)


,skuNew,offerTermCode,sku,effectiveDate,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,A53K5SWT3G3CRHF5,JRTCKXETXF,A53K5SWT3G3CRHF5,2026-05-01T00:00:00Z,A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7,$0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia),0,Inf,hours,0.2734800000,NaN
1,TCP2YWGAHGPW67N9,JRTCKXETXF,TCP2YWGAHGPW67N9,2026-05-01T00:00:00Z,TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7,$0.47002 per hour for EKS Auto Mode management of m6g.metal in South America (Sao Paulo),0,Inf,hours,0.4700200000,NaN
2,MCK76ZZJZMWS7HDE,JRTCKXETXF,MCK76ZZJZMWS7HDE,2026-05-01T00:00:00Z,MCK76ZZJZMWS7HDE.JRTCKXETXF.6YS6EN2CT7,$0.04224 per hour for EKS Auto Mode management of r5d.xlarge in Middle East (Bahrain),0,Inf,hours,0.0422400000,NaN
3,FQKX5BN95S5F5T2E,JRTCKXETXF,FQKX5BN95S5F5T2E,2026-05-01T00:00:00Z,FQKX5BN95S5F5T2E.JRTCKXETXF.6YS6EN2CT7,$0.11520 per hour for EKS Auto Mode management of m5.4xlarge in Asia Pacific (Sydney),0,Inf,hours,0.1152000000,NaN
4,USZRSB47YGRTASDR,JRTCKXETXF,USZRSB47YGRTASDR,2026-05-01T00:00:00Z,USZRSB47YGRTASDR.JRTCKXETXF.6YS6EN2CT7,$0.03074 per hour for EKS Auto Mode management of i8ge.large in Asia Pacific (Malaysia),0,Inf,hours,0.0307400000,NaN


2. Αρχικά παρατηρούμε 2 στήλες με το sku της υπηρεσίας. Η μία ήταν εξαρχής μέσα στα δεδομένα τιμολόγησης με την ονομασία sku, και η άλλη προστέθηκε κατά το άνοιγμα των υπηρεσιών. Συγκεκριμένα όλα τα δεδομένα terms είχαν σαν αρχικό αναγνωριστικό το sku χύμα, και μετά τα δεδομένα: κάπως έτσι "xxx: {dict with pricing info}". Οπότε το αρχικό κλειδί το κάναμε στήλη. Τώρα θα γράψουμε κώδικα ο οποίος ελέγχει αν υπάρχει ταύτιση skuNew με sku, αν δεν υπάρχει θα πετάει την εγγραφή, και στο τέλος θα αφαιρεί μία από τις δύο στήλες.

In [ ]:
rowCount = len(df_terms_final)

#Βάζω το if για να μπορώ να τρέχω το κελί και μόνο του χωρίς να πετάει error
if 'skuNew' in df_terms_final.columns and 'sku' in df_terms_final.columns:

    # Φτιάχνω την συνθήκη ελέγχου - διαγραφής μιας υπηρεσίας και την εφαρμόζω απευθείας μετά πάνω στο dataframe. Γλιτώνω το loop 
    condition = (df_terms_final['skuNew'] == df_terms_final['sku'])

    df_filtered_terms = df_terms_final[condition]

    df_terms_final = df_filtered_terms.copy()

print(f"Initial records: {rowCount}")
print(f"Rejected records: {rowCount - len(df_terms_final)}")

if 'skuNew' in df_terms_final.columns:
    df_terms_final = df_terms_final.drop(columns=['skuNew'])

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Initial records: 20530
Rejected records: 0
Dimensions (rows,cols): (20530, 10)


,offerTermCode,sku,effectiveDate,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,JRTCKXETXF,A53K5SWT3G3CRHF5,2026-05-01T00:00:00Z,A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7,$0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia),0,Inf,hours,0.2734800000,NaN
1,JRTCKXETXF,TCP2YWGAHGPW67N9,2026-05-01T00:00:00Z,TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7,$0.47002 per hour for EKS Auto Mode management of m6g.metal in South America (Sao Paulo),0,Inf,hours,0.4700200000,NaN
2,JRTCKXETXF,MCK76ZZJZMWS7HDE,2026-05-01T00:00:00Z,MCK76ZZJZMWS7HDE.JRTCKXETXF.6YS6EN2CT7,$0.04224 per hour for EKS Auto Mode management of r5d.xlarge in Middle East (Bahrain),0,Inf,hours,0.0422400000,NaN
3,JRTCKXETXF,FQKX5BN95S5F5T2E,2026-05-01T00:00:00Z,FQKX5BN95S5F5T2E.JRTCKXETXF.6YS6EN2CT7,$0.11520 per hour for EKS Auto Mode management of m5.4xlarge in Asia Pacific (Sydney),0,Inf,hours,0.1152000000,NaN
4,JRTCKXETXF,USZRSB47YGRTASDR,2026-05-01T00:00:00Z,USZRSB47YGRTASDR.JRTCKXETXF.6YS6EN2CT7,$0.03074 per hour for EKS Auto Mode management of i8ge.large in Asia Pacific (Malaysia),0,Inf,hours,0.0307400000,NaN


3. Η στήλη effectiveDate είναι σε ίδιο μήκος κύματος με τις στήλες που έχει η azure, και η google στα δεδομένα της. Περιγράφει την ημερομηνία και ώρα εκκίνησης της συγκεκριμένης τιμής που υπάρχει για την υπηρεσία. Στην εργασία δεν μας ενδιαφέρει η ιστορικότητα των δεδομέων, οπότε την αφαιρούμε απευθείας με τις αντίστοιχες εντολές.

In [74]:
if 'effectiveDate' in df_terms_final.columns:
    df_terms_final = df_terms_final.drop(columns=['effectiveDate'])

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (20530, 9)


,offerTermCode,sku,rateCode,description,beginRange,endRange,unit,priceUSD,appliesTo
0,JRTCKXETXF,A53K5SWT3G3CRHF5,A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7,$0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia),0,Inf,hours,0.2734800000,NaN
1,JRTCKXETXF,TCP2YWGAHGPW67N9,TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7,$0.47002 per hour for EKS Auto Mode management of m6g.metal in South America (Sao Paulo),0,Inf,hours,0.4700200000,NaN
2,JRTCKXETXF,MCK76ZZJZMWS7HDE,MCK76ZZJZMWS7HDE.JRTCKXETXF.6YS6EN2CT7,$0.04224 per hour for EKS Auto Mode management of r5d.xlarge in Middle East (Bahrain),0,Inf,hours,0.0422400000,NaN
3,JRTCKXETXF,FQKX5BN95S5F5T2E,FQKX5BN95S5F5T2E.JRTCKXETXF.6YS6EN2CT7,$0.11520 per hour for EKS Auto Mode management of m5.4xlarge in Asia Pacific (Sydney),0,Inf,hours,0.1152000000,NaN
4,JRTCKXETXF,USZRSB47YGRTASDR,USZRSB47YGRTASDR.JRTCKXETXF.6YS6EN2CT7,$0.03074 per hour for EKS Auto Mode management of i8ge.large in Asia Pacific (Malaysia),0,Inf,hours,0.0307400000,NaN


4. Κλιμακωτές χρεώσεις!! Όλοι οι πάροχοι τις έχουν και σε όλους τις παραλέιπω και κρατάω μόνο την πρώτη βαθμίδα χρέωσης. Το ίδιο και εδω!! Η aws αναπαριστά τις κλιμακωτές χρεώσεις μέσω των πεδίων beginRange και endRange τα οποία καθορίζουν τα όρια. Το 0 στο beginRange είναι αυτό το οποίο μας ενδιαφέρει καθώς είναι η πρώτη - βασική βαθμίδα χρέσωσης. Οπότε θα κρατήσουμε όλες τις εγγραφές εκείνες που έχουν beginRange == 0, και μετά τις στήλες με τα όρια θα τις αφαιρέσουμε, αφού καμία σημασία δεν θα έχουν πλέον.
2 παρατηρήσεις:
- Η aws διατηρεί το ίδιο sku ανάμεσα στις κλίμακες, αλλά αλλάζει το rateCode και συγκεκριμένα τα τελευταία του ψηφία
- Το offerTermCode παρατηρούμε ότι είναι ίδιο για πολλες εγγραφές

In [75]:
tierRates = df_terms_final['beginRange'].unique()
print(tierRates)

['0' '576000' '1152000' '5760000' '11520000']


In [76]:
if 'beginRange' in df_terms_final.columns and 'endRange' in df_terms_final.columns:

    df_terms_final = df_terms_final[df_terms_final['beginRange'] == '0']

    df_terms_final.drop(columns=['beginRange', 'endRange'], inplace=True)

df_terms_final.reset_index(drop=True, inplace=True)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (20394, 7)


,offerTermCode,sku,rateCode,description,unit,priceUSD,appliesTo
0,JRTCKXETXF,A53K5SWT3G3CRHF5,A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7,$0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia),hours,0.2734800000,NaN
1,JRTCKXETXF,TCP2YWGAHGPW67N9,TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7,$0.47002 per hour for EKS Auto Mode management of m6g.metal in South America (Sao Paulo),hours,0.4700200000,NaN
2,JRTCKXETXF,MCK76ZZJZMWS7HDE,MCK76ZZJZMWS7HDE.JRTCKXETXF.6YS6EN2CT7,$0.04224 per hour for EKS Auto Mode management of r5d.xlarge in Middle East (Bahrain),hours,0.0422400000,NaN
3,JRTCKXETXF,FQKX5BN95S5F5T2E,FQKX5BN95S5F5T2E.JRTCKXETXF.6YS6EN2CT7,$0.11520 per hour for EKS Auto Mode management of m5.4xlarge in Asia Pacific (Sydney),hours,0.1152000000,NaN
4,JRTCKXETXF,USZRSB47YGRTASDR,USZRSB47YGRTASDR.JRTCKXETXF.6YS6EN2CT7,$0.03074 per hour for EKS Auto Mode management of i8ge.large in Asia Pacific (Malaysia),hours,0.0307400000,NaN


5. Μηδενικές χρεώσεις. Δεν μας ενδιαφέρουν, αποτελούν εξαιρέσεις και ειδικές περιπτώσεις για αυτό και θα τις φιλτράρουμε. Η αιτιολόγηση βρίσκεται στο report και ακολουθεί την ίδια στρατηγική με τους άλλους παρόχους, όπου επίσης απορρίψαμε δωρεάν εγγραφές. Κάνουμε και την στήλη floar για καλύτερες μαθηματικές πράξεις

In [77]:
pd.set_option('display.max_colwidth', None)
df_terms_final['description'].head(5)

0    $0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia)
1    $0.47002 per hour for EKS Auto Mode management of m6g.metal in South America (Sao Paulo)
2       $0.04224 per hour for EKS Auto Mode management of r5d.xlarge in Middle East (Bahrain)
3       $0.11520 per hour for EKS Auto Mode management of m5.4xlarge in Asia Pacific (Sydney)
4     $0.03074 per hour for EKS Auto Mode management of i8ge.large in Asia Pacific (Malaysia)
Name: description, dtype: object

In [78]:
df_terms_final['priceUSD'] = pd.to_numeric(df_terms_final['priceUSD'], errors='coerce')

df_terms_final = df_terms_final[df_terms_final['priceUSD'] > 0]

df_terms_final.reset_index(drop=True, inplace=True)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (20394, 7)


,offerTermCode,sku,rateCode,description,unit,priceUSD,appliesTo
0,JRTCKXETXF,A53K5SWT3G3CRHF5,A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7,$0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia),hours,0.27348,NaN
1,JRTCKXETXF,TCP2YWGAHGPW67N9,TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7,$0.47002 per hour for EKS Auto Mode management of m6g.metal in South America (Sao Paulo),hours,0.47002,NaN
2,JRTCKXETXF,MCK76ZZJZMWS7HDE,MCK76ZZJZMWS7HDE.JRTCKXETXF.6YS6EN2CT7,$0.04224 per hour for EKS Auto Mode management of r5d.xlarge in Middle East (Bahrain),hours,0.04224,NaN
3,JRTCKXETXF,FQKX5BN95S5F5T2E,FQKX5BN95S5F5T2E.JRTCKXETXF.6YS6EN2CT7,$0.11520 per hour for EKS Auto Mode management of m5.4xlarge in Asia Pacific (Sydney),hours,0.11520,NaN
4,JRTCKXETXF,USZRSB47YGRTASDR,USZRSB47YGRTASDR.JRTCKXETXF.6YS6EN2CT7,$0.03074 per hour for EKS Auto Mode management of i8ge.large in Asia Pacific (Malaysia),hours,0.03074,NaN


5. Σχετικά με την στήλη appliesTo δηλώνει εξάρτηση μεταξ`ύ 2 υπηρεσιών. Δηλαδή εάν η υπηρεσία που βλέπουμε εξαρτάται από κάποια άλλη. Δεν θέλω εξαρτήσεις, άρα υπηρεσίες που δεν έχουν Nan στην στήλη αυτή απορρίπτονται, και κατόπιν απορρίπτεται και η στήλη αυτή καθ'αυτή

In [79]:
apTo = df_terms_final['appliesTo'].unique()
print(apTo)

[nan]


In [80]:
if 'appliesTo' in df_terms_final.columns:
    df_terms_final = df_terms_final[df_terms_final['appliesTo'].isna()]

    df_terms_final.reset_index(drop=True, inplace=True)

    df_terms_final.drop(columns=['appliesTo'], errors='ignore', inplace=True)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (20394, 6)


,offerTermCode,sku,rateCode,description,unit,priceUSD
0,JRTCKXETXF,A53K5SWT3G3CRHF5,A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7,$0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia),hours,0.27348
1,JRTCKXETXF,TCP2YWGAHGPW67N9,TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7,$0.47002 per hour for EKS Auto Mode management of m6g.metal in South America (Sao Paulo),hours,0.47002
2,JRTCKXETXF,MCK76ZZJZMWS7HDE,MCK76ZZJZMWS7HDE.JRTCKXETXF.6YS6EN2CT7,$0.04224 per hour for EKS Auto Mode management of r5d.xlarge in Middle East (Bahrain),hours,0.04224
3,JRTCKXETXF,FQKX5BN95S5F5T2E,FQKX5BN95S5F5T2E.JRTCKXETXF.6YS6EN2CT7,$0.11520 per hour for EKS Auto Mode management of m5.4xlarge in Asia Pacific (Sydney),hours,0.11520
4,JRTCKXETXF,USZRSB47YGRTASDR,USZRSB47YGRTASDR.JRTCKXETXF.6YS6EN2CT7,$0.03074 per hour for EKS Auto Mode management of i8ge.large in Asia Pacific (Malaysia),hours,0.03074


6. Κοιτάζουμε σε ορισμένες κρίσιμες στήλες, για τιμές με Nan προκειμένου να αφαιρέσουμε τις αντίστοιχες εγγραφές. Με βάση την κοινή λογική θα εξετάσουμε τις στήλες της τιμής, της μονάδας μέτρησης, και του sku, καθώς έαν σε μία από αυτές τις κρίσιμες στήλες απουσιάζει μία τιμή, τα δεδομένα δεν θα μπορούν να αναλυθούν

In [81]:
colsToCheck = ['sku', 'priceUSD', 'unit']
before = len(df_terms_final)

df_terms_final.dropna(subset=colsToCheck, inplace=True)

if before - len(df_terms_final) > 0:
    print(f"Removed {before - len(df_terms_final)} rows due to missing critical values.")

df_terms_final.reset_index(drop=True, inplace=True)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (20394, 6)


,offerTermCode,sku,rateCode,description,unit,priceUSD
0,JRTCKXETXF,A53K5SWT3G3CRHF5,A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7,$0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia),hours,0.27348
1,JRTCKXETXF,TCP2YWGAHGPW67N9,TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7,$0.47002 per hour for EKS Auto Mode management of m6g.metal in South America (Sao Paulo),hours,0.47002
2,JRTCKXETXF,MCK76ZZJZMWS7HDE,MCK76ZZJZMWS7HDE.JRTCKXETXF.6YS6EN2CT7,$0.04224 per hour for EKS Auto Mode management of r5d.xlarge in Middle East (Bahrain),hours,0.04224
3,JRTCKXETXF,FQKX5BN95S5F5T2E,FQKX5BN95S5F5T2E.JRTCKXETXF.6YS6EN2CT7,$0.11520 per hour for EKS Auto Mode management of m5.4xlarge in Asia Pacific (Sydney),hours,0.11520
4,JRTCKXETXF,USZRSB47YGRTASDR,USZRSB47YGRTASDR.JRTCKXETXF.6YS6EN2CT7,$0.03074 per hour for EKS Auto Mode management of i8ge.large in Asia Pacific (Malaysia),hours,0.03074


7. Η στήλη offerTermCode μετά από εξέταση 5 διαφορετικών υπηρεσιών τιμολόγησης της aws ,φαίνεται να περιέχει τον ίδιο ακριβώς κωδικό. Αυτό διότι το offerTermCode είναι ο παγκόσμιος κωδικός τύπου σύμβασης. Εμείς έχουμε κρατήση σε όλους το On-Demand, οπότε θα είναι πάντα ίδιο. Βάση αυτού η στήλη δεν κρίνεται απαραίτητη στο τελικό dataset και άρα την απορρίπτουμε

In [82]:
df_terms_final['offerTermCode'].unique()

array(['JRTCKXETXF'], dtype=object)

In [83]:
if 'offerTermCode' in df_terms_final.columns:

    df_terms_final.drop(columns=['offerTermCode'], errors='ignore', inplace=True)

print(f"Dimensions (rows,cols): {df_terms_final.shape}")
df_terms_final.head()

Dimensions (rows,cols): (20394, 5)


,sku,rateCode,description,unit,priceUSD
0,A53K5SWT3G3CRHF5,A53K5SWT3G3CRHF5.JRTCKXETXF.6YS6EN2CT7,$0.27348 per hour for EKS Auto Mode management of r8i.8xlarge in Asia Pacific (Malaysia),hours,0.27348
1,TCP2YWGAHGPW67N9,TCP2YWGAHGPW67N9.JRTCKXETXF.6YS6EN2CT7,$0.47002 per hour for EKS Auto Mode management of m6g.metal in South America (Sao Paulo),hours,0.47002
2,MCK76ZZJZMWS7HDE,MCK76ZZJZMWS7HDE.JRTCKXETXF.6YS6EN2CT7,$0.04224 per hour for EKS Auto Mode management of r5d.xlarge in Middle East (Bahrain),hours,0.04224
3,FQKX5BN95S5F5T2E,FQKX5BN95S5F5T2E.JRTCKXETXF.6YS6EN2CT7,$0.11520 per hour for EKS Auto Mode management of m5.4xlarge in Asia Pacific (Sydney),hours,0.11520
4,USZRSB47YGRTASDR,USZRSB47YGRTASDR.JRTCKXETXF.6YS6EN2CT7,$0.03074 per hour for EKS Auto Mode management of i8ge.large in Asia Pacific (Malaysia),hours,0.03074


Τέλος ανάλυσης πίνακα terms!! 
Επειδή το notebook έχει ήδη μεγαλώσει και δεν θέλω η ανάλυση του products να γίνει εδώ αποθηκεύουμε τοπικά στο current directory τα 2 datframe τα οποία επεξεργαζόμαστε (εκείνο με τα προϊόντα και το καθαρισμένο terms), και συνεχίζουμε σε άλλο notebook την ανάλυση του products και το merge

In [84]:
df_terms_final.to_parquet('cleaned_aws_terms.parquet')
df_products.to_parquet('cleaned_aws_products.parquet')
print("Saved successfully!")

Saved successfully!
